# DP2 — DiaObject / DiaSource catalog query over a DDF via Butler

**Author:** dagoret  
**Creation Date:** 2026-03-13  
**Last Date:** 2026-03-17  
**version:** v1
## Purpose

Retrieve **DiaObject**, **DiaSource**, and **ForcedSourceOnDiaObject** catalogs
for a user-selected Deep Drilling Field (DDF) using **Butler Gen3 only**
(no TAP service, not yet available for DP2 at USDF).

## Actual schema (discovered at runtime — COSMOS, LSSTCam/runs/DRP/20250417_20250921/w_2025_49/DM-53545)

- `dia_object` : dims `(skymap, tract)` — 1 ref per tract, index = `diaObjectId`
- `dia_source` : dims `(skymap, tract)` — 1 ref per tract, index = `diaSourceId` (join on `diaObjectId` column)
- `dia_object_forced_source` : dims `(skymap, tract, patch)` — **21 refs per tract** (one per patch)
  - Must iterate over all patch refs and concat
  - Contains per-visit forced photometry linked to DiaObjects

## Reference notebooks
- `2026-03-07_AccessToDP2.ipynb` — Butler setup
- `2026-03-13_DP2_DDF_Tracts_SkyMap_Mollweide.ipynb` — tract lookup

---
## 0. Imports

In [ ]:
import warnings

warnings.filterwarnings("ignore")
import traceback
import gc

import os

import numpy as np
import pandas as pd
import matplotlib
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

from astropy.coordinates import SkyCoord
from astropy.table import Table
import astropy.units as u
from astropy.time import Time
from astropy.coordinates import Angle

from lsst.daf.butler import Butler
from lsst.geom import SpherePoint, degrees

print(f"Matplotlib : {matplotlib.__version__}")
print(f"NumPy      : {np.__version__}")
print(f"Pandas     : {pd.__version__}")
print("Imports OK")

In [ ]:
mpl.rcParams.update(
    {
        "figure.figsize": (8, 5),  # taille par défaut des figures
        "font.size": 14,  # taille de base
        "axes.titlesize": 18,  # titre de l'axe
        "axes.labelsize": 16,  # labels x et y
        "xtick.labelsize": 14,  # ticks x
        "ytick.labelsize": 14,  # ticks y
        "legend.fontsize": 14,  # texte de la légende
        "legend.title_fontsize": 15,  # titre de la légende
        "figure.titlesize": 20,  # titre global de la figure
    }
)

In [ ]:
# Remove to run faster the notebook
# import ipywidgets as widgets
# %matplotlib widget

# Enable interactive matplotlib backend with zoom/pan toolbar
# Requires: pip install ipympl
# If ipympl is not available, fall back to inline (no interactivity)
try:
    import ipympl  # noqa: F401

    %matplotlib widget
    print("ipympl found → interactive backend (%matplotlib widget)")
except ImportError:
    %matplotlib inline
    print("ipympl NOT found → falling back to %matplotlib inline (no zoom widget)")
    print("Install with:  pip install ipympl")

In [ ]:
# output dirs
NB_TAG = "DP2_DDF_DIAOBJ_BUTLER_01"
DIR_DATA = f"data_{NB_TAG}"
DIR_FIGS = f"figs_{NB_TAG}"
os.makedirs(DIR_DATA, exist_ok=True)
os.makedirs(DIR_FIGS, exist_ok=True)
print(f"Data : {os.path.abspath(DIR_DATA)}")
print(f"Figs : {os.path.abspath(DIR_FIGS)}")

# Output directory for DRP data
OUTPUT_DIR = DIR_DATA
if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

# Visits filename
FILENAME_VISITS = "visitTable-2025041500138-2026010800515_N34276.csv"

In [ ]:
# debug flags
DEBUG = False
DEBUG_VISITS = False

In [ ]:
# FLAGS
FLAG_FETCH_VISITSEXPOSURES = False

## Usefull functions

- from https://github.com/sylvielsstfr/mySIT-COM2025/blob/main/notebooks/2025-05-30-Visits/05_SourcesFromVisits.ipynb

In [ ]:
def getLostOfVisits(registry, where_clause_date):
    """
    Get exposure/visits from butler registry
    parameters:
    ==========
        registry : butler registry
        where_clause_date : specify which instrument and exposure dates in the butler registry
    """
    df_exposure = pd.DataFrame(
        {
            "id": pd.Series(dtype="int"),
            "obs_id": pd.Series(dtype="int"),
            "day_obs": pd.Series(dtype="int"),
            "seq_num": pd.Series(dtype="int"),
            "time_start": pd.Series(dtype="str"),  # ou 'datetime64[ns]' si c'est un datetime
            "time_end": pd.Series(dtype="str"),  # idem
            "type": pd.Series(dtype="str"),
            "target": pd.Series(dtype="str"),
            "filter": pd.Series(dtype="str"),
            "zenith_angle": pd.Series(dtype="float"),
            "expos": pd.Series(dtype="float"),  # ou 'int' selon le cas
            "ra": pd.Series(dtype="float"),
            "dec": pd.Series(dtype="float"),
            "skyangle": pd.Series(dtype="float"),
            "azimuth": pd.Series(dtype="float"),
            "zenith": pd.Series(dtype="float"),
            "science_program": pd.Series(dtype="str"),
            "jd": pd.Series(dtype="float"),
            "mjd": pd.Series(dtype="float"),
        }
    )

    # save the data array in rows before saving in pandas dataframe
    rows = []
    for count, info in enumerate(registry.queryDimensionRecords("exposure", where=where_clause_date)):
        try:
            jd_start = info.timespan.begin.value
            jd_end = info.timespan.end.value
            the_Time_start = Time(jd_start, format="jd", scale="utc")
            the_Time_end = Time(jd_end, format="jd", scale="utc")
            mjd_start = the_Time_start.mjd
            mjd_end = the_Time_end.mjd
            isot_start = the_Time_start.isot
            isot_end = the_Time_end.isot

            if count == 0:
                print("===== Time Conversion Debug Info =====")
                print(f"JD start      : {jd_start} (type: {type(jd_start)})")
                print(f"JD end        : {jd_end} (type: {type(jd_end)})")
                print(f"MJD start     : {mjd_start} (type: {type(mjd_start)})")
                print(f"MJD end       : {mjd_end} (type: {type(mjd_end)})")
                print(f"ISOT start    : {isot_start} (type: {type(isot_start)})")
                print(f"ISOT end      : {isot_end} (type: {type(isot_end)})")
                print("=======================================")

            # put row in a dictionnary before stacking
            row = {
                "id": info.id,
                "obs_id": info.obs_id,
                "day_obs": info.day_obs,
                "seq_num": info.seq_num,
                "time_start": isot_start,
                "time_end": isot_end,
                "type": info.observation_type,
                "target": info.target_name,
                "filter": info.physical_filter,
                "zenith_angle": info.zenith_angle,
                "expos": info.exposure_time,  # Exemple : adapter selon ton objet
                "ra": info.tracking_ra,
                "dec": info.tracking_dec,
                "skyangle": info.sky_angle,
                "azimuth": info.azimuth,
                "zenith": info.zenith_angle,
                "science_program": info.science_program,
                "jd": float(jd_start),
                "mjd": float(mjd_start),
            }
            rows.append(row)

        except ValueError as e:
            print(f"Erreur de valeur : {e}")
        except FileNotFoundError as e:
            print(f"Fichier introuvable : {e}")
        except Exception as e:
            print(f"Erreur inattendue : {type(e).__name__} - {e}")
            print(f">>>   Unexpected error at row {count}:", sys.exc_info()[0])
            traceback.print_exc()  # affiche la stack trace complète

    # Création finale du DataFrame
    df_exposure = pd.DataFrame(rows)
    df_science = df_exposure[df_exposure.type == "science"]
    df_science.reset_index(drop=True, inplace=True)

    return df_science

In [ ]:
def FetchTimesForVisits(visit_list):
    """
    Keep for example and possible later use
    """
    # On interroge la table visitDefinition
    Nvisit = len(visit_list)

    if Nvisit == 1:
        thevisit = visit_list.values[0]
        rows = registry.queryDimensionRecords("visit", where=f"visit={thevisit}")
    else:
        rows = registry.queryDimensionRecords("visit", where=f"visit in {tuple(visit_list)}")

    # 4. Construire un tableau des résultats
    results = []
    for row in rows:
        visit_id = row.id
        visit_airmass = 1.0 / np.cos(Angle(row.zenith_angle, u.degree).rad)
        visit_azimuth = row.azimuth

        # Extraire l'instant de début de l'observation (Time astropy)
        start_time = row.timespan.begin

        # Convertir en MJD et ISO
        mjd = start_time.to_value("mjd")  # Ex: 60384.28718
        isot = start_time.to_value("isot")  # Ex: '2024-04-19 06:53:32.000'

        # mjd = row.startDate.toMjd()
        # utc = Time(mjd, format='mjd', scale='utc').to_value('iso')
        # results.append({"visit": visit_id, "mjd": mjd, "isot": isot})
        results.append(
            {"visit": visit_id, "mjd": mjd, "isot": isot, "airmass": visit_airmass, "azimuth": visit_azimuth}
        )

    df_times = pd.DataFrame(results).sort_values("visit")
    df_times.set_index("visit", inplace=True)
    return df_times

In [ ]:
def RetrieveDRPSources_forTarget(butler, center_coord, datasettype, where_clause, radius_cut=50):
    """
    Keep for example and possible later use
    Find the closest Sourcesthe target_coord

    parameters:
    - butler
    - the coordinate of the target (center of the cone seach)
    - the datasettype name for the DRP object
    - where_clause : which contrain requirements on the tract and patch numbers
    - cut on angluar separation for the returned for the returned object

    Return
    - object Id with minimum separation ,
    - minimum separation (arcec),
    - the table of DRP objects within the radius_cut
    """

    ra_columns = ["u_ra", "g_ra", "r_ra", "i_ra", "z_ra", "y_ra"]
    dec_columns = ["u_dec", "g_dec", "r_dec", "i_dec", "z_dec", "y_dec"]

    therefs = butler.registry.queryDatasets(datasettype, collections=collection, where=where_clause)

    for count, ref in enumerate(therefs):
        the_id = ref.dataId
        the_tract_id = the_id["tract"]
        print(the_id)

        # catalog of rubin objects (a pandas Dataframe) inside the tract
        catalog = butler.get(ref)
        catalog = catalog[catalog["patch"] == patchNbSel]

        nobjects = len(catalog)

        # Calcul de la moyenne ligne par ligne, en ignorant les NaN
        catalog["ra"] = catalog[ra_columns].mean(axis=1, skipna=True)
        catalog["dec"] = catalog[dec_columns].mean(axis=1, skipna=True)

        # extract the (ra,dec) coordinates for all te objects of the rubin-catalog
        ra_cat = catalog["ra"].values
        dec_cat = catalog["dec"].values
        # coordinates for all rubin-catalog points
        catalog_coords = SkyCoord(ra=ra_cat * u.deg, dec=dec_cat * u.deg)

        # Angular distance to target
        distances_arcsec = center_coord.separation(catalog_coords).arcsecond

        # add the separation angle to the ctalog
        catalog["sep"] = distances_arcsec

        # closest object from the target
        sepMin = distances_arcsec.min()
        sepMin_idx = np.where(distances_arcsec == sepMin)[0][0]

        closest_obj = catalog[catalog["sep"] <= sepMin]

        # select a few of these sources to debug the closest candidate
        nearby_obj = catalog[distances_arcsec < radius_cut]

        return closest_obj, sepMin, nearby_obj

In [ ]:
# =========================================================================
# Helper: add a top x-axis showing calendar dates above the MJD axis
# =========================================================================
# The bottom x-axis of each light curve panel uses MJD.
# This helper adds a twin axis on top with evenly-spaced date labels
# (UTC) so you can immediately read off known observing dates.
#
# Usage (call AFTER plotting, BEFORE tight_layout / savefig):
#   add_date_top_axis(axes[0], n_ticks=6)
# =========================================================================


def add_date_top_axis(ax, n_ticks=6, date_fmt="%Y-%m-%d"):
    """
    Add a secondary x-axis on top of `ax` showing calendar dates.

    The secondary axis shares the same MJD limits as the primary axis
    but labels ticks as UTC calendar dates (e.g. 2025-06-15).

    Parameters
    ----------
    ax : matplotlib Axes
        Primary axes whose bottom x-axis carries MJD values.
    n_ticks : int
        Number of evenly-spaced tick marks on the top axis.
    date_fmt : str
        strftime format for the date labels (default YYYY-MM-DD).

    Returns
    -------
    ax_top : matplotlib Axes
        The new twin axes placed on top.
    """
    mjd_min, mjd_max = ax.get_xlim()

    # Build evenly-spaced MJD tick positions spanning the current x range
    mjd_ticks = np.linspace(mjd_min, mjd_max, n_ticks)

    # Convert each MJD tick to a calendar date string via astropy
    date_labels = [Time(m, format="mjd", scale="utc").to_value("isot")[:10] for m in mjd_ticks]

    # Create a twin axis that shares the same x-scale
    ax_top = ax.twiny()
    ax_top.set_xlim(mjd_min, mjd_max)  # must match primary axis exactly
    ax_top.set_xticks(mjd_ticks)
    ax_top.set_xticklabels(date_labels, rotation=30, ha="left", fontsize=8)
    ax_top.set_xlabel("Date (UTC)", fontsize=9, labelpad=4)

    return ax_top


print("add_date_top_axis() helper defined.")

In [ ]:
def load_visits_file(file_path):
    """
    Checks if the visits file exists and loads it into a DataFrame.
    """
    if not os.path.exists(file_path):
        print(f"ERROR: File not found at {file_path}")
        return None

    try:
        # Standard LSST tables are usually Parquet
        if file_path.endswith(".parquet"):
            df_visits = pd.read_parquet(file_path)
        else:
            df_visits = pd.read_csv(file_path)

        print(f"Successfully loaded {len(df_visits):,} visits from {os.path.basename(file_path)}")
        return df_visits
    except Exception as e:
        print(f"ERROR loading file: {e}")
        return None


# --- Usage ---
# df_visits = load_visits_file(FULLFILENAME_VISITS)

---
## 1. User Configuration

In [ ]:
SELECTED_DDF = "COSMOS"  # COSMOS | ECDFS | ELAIS-S1 | XMM-LSS | EDFS-a | EDFS-b | EDFS | M49
CONE_RADIUS_DEG = 1.8
MIN_NSOURCES = 200

REPO = "dp2_prep"
COLLECTION = "LSSTCam/runs/DRP/20250417_20250921/w_2025_49/DM-53545"
# COLLECTION  = "LSSTCam/runs/DRP/v30_0_0/DM-53877" # No Skymap
SKYMAP_NAME = "lsst_cells_v2"
INSTRUMENT = "LSSTCam"
WHERE_CLAUSE_INSTRUMENT = "instrument = '" + f"{INSTRUMENT}" + "'"
DATE_START = 20250415
WHERE_CLAUSE_DATE = WHERE_CLAUSE_INSTRUMENT + f" and day_obs >= {DATE_START}"


TRACT_SEARCH_RADIUS_DEG = 1.8

DDF_COORDS = {
    "COSMOS": (150.119, +2.206),
    "ECDFS": (53.125, -28.100),
    "ELAIS-S1": (9.450, -44.000),
    "XMM-LSS": (35.708, -4.750),
    "EDFS-a": (58.900, -49.315),
    "EDFS-b": (63.600, -47.600),
    "EDFS": (61.240, -48.423),
    "M49": (187.400, +8.000),
}

RA_CENTER, DEC_CENTER = DDF_COORDS[SELECTED_DDF]
print(f"DDF         : {SELECTED_DDF}  ({RA_CENTER:.4f}°, {DEC_CENTER:+.4f}°)")
print(f"Cone radius : {CONE_RADIUS_DEG}°")
print(f"Min nDiaSrc : {MIN_NSOURCES}")
print(f"Collection  : {COLLECTION}")
print(f"Instrument  : {INSTRUMENT}")
print(f"Where clause  : {WHERE_CLAUSE_INSTRUMENT}")
print(f"Where clause date : {WHERE_CLAUSE_DATE}")

---
## 2. Butler and SkyMap

In [ ]:
butler = Butler(REPO, collections=COLLECTION)
registry = butler.registry
print("Butler OK")

In [ ]:
try:
    skymap = butler.get("skyMap", skymap=SKYMAP_NAME, collections=COLLECTION)
except Exception:
    skymap = butler.get("skyMap", skymap=SKYMAP_NAME)
print(f"SkyMap '{SKYMAP_NAME}' : {len(skymap)} tracts")

### 2.1 Search for visit tables

In [ ]:
visitTable_pattern1 = "*isitTable*"
visitTable_pattern2 = "*isitTable"
visitTable_name = "preVisitTable"

In [ ]:
if DEBUG_VISITS:
    dataset_types = registry.queryDatasetTypes(visitTable_pattern1)
    for dt in dataset_types:
        print(dt.name)

In [ ]:
if DEBUG_VISITS:
    dataset_types = list(butler.registry.queryDatasetTypes(visitTable_pattern1))

    # Pour un affichage lisible (nom du type et storage class)
    for dt in sorted(dataset_types, key=lambda x: x.name):
        print(f"{dt.name:20} | {dt.storageClass.name}")

In [ ]:
if DEBUG_VISITS:
    dataset_types = list(butler.registry.queryDatasetTypes(visitTable_pattern2))
    for dt in dataset_types:
        print(f"{dt.name} :::", dt)
        print(f"    required dimensions: {dt.dimensions.required}")
        print()

## 2.2 Retrieve the science visits in order to match the visitid with the MJD in forced source photometry
- `visitid = id`
- `MJD = mjd `

In [ ]:
if FLAG_FETCH_VISITSEXPOSURES:
    visitTable = getLostOfVisits(registry, where_clause_date=WHERE_CLAUSE_DATE)
    print(visitTable.head(1))
    visitMin, visitMax, Nvisits = visitTable["id"].min(), visitTable["id"].max(), len(visitTable)
    print(f"visitMin = {visitMin},visitMax = {visitMax}, Nvisits = {Nvisits}")
    visit_filename = f"visitTable-{visitMin}-{visitMax}_N{Nvisits}.csv"
    print(f"filename_visits = {visit_filename}")
    visit_fullfilename = os.path.join(OUTPUT_DIR, visit_filename)
    visitTable.to_csv(visit_fullfilename)
    del visitTable
    gc.collect()

In [ ]:
# visitTable = getLostOfVisits(registry, where_clause_date=WHERE_CLAUSE_DATE)

In [ ]:
# print(visitTable.head(1))

---
## 3. Dataset Type Inventory

Identify the correct names and **dimensions** for DIA products — in particular
`dia_object_forced_source` has `(skymap, tract, patch)` dimensions, which
requires iterating over patch refs.

In [ ]:
# Hard-coded after discovery: confirmed present in DM-54249
DIA_OBJ_TYPE = "dia_object"  # dims: skymap, tract
DIA_SRC_TYPE = "dia_source"  # dims: skymap, tract
FORCED_DIA_TYPE = "dia_object_forced_source"  # dims: skymap, tract, patch

# Print actual dimensions for confirmation
for dt_name in [DIA_OBJ_TYPE, DIA_SRC_TYPE, FORCED_DIA_TYPE]:
    try:
        dt = registry.getDatasetType(dt_name)
        in_col = registry.queryDatasets(dt_name, collections=COLLECTION).any(execute=False, exact=False)
        print(f"{dt_name:40s}  dims={list(dt.dimensions.names)}  present={in_col}")
    except Exception as exc:
        print(f"{dt_name:40s}  ERROR: {exc}")

---
## 4. Find Tracts Covering the DDF

In [ ]:
def find_tracts_for_coord(skymap, ra_deg, dec_deg, radius_deg=1.8):
    cos_dec = max(np.cos(np.deg2rad(dec_deg)), 0.01)
    step = 0.35
    found_ids = set()
    for ddec in np.arange(-radius_deg, radius_deg + step, step):
        for dra in np.arange(-radius_deg, radius_deg + step, step):
            if np.sqrt(dra**2 + ddec**2) > radius_deg:
                continue
            ra_s = ra_deg + dra / cos_dec
            dec_s = dec_deg + ddec
            if not (-89.9 <= dec_s <= 89.9):
                continue
            sp = SpherePoint(ra_s * degrees, dec_s * degrees)
            try:
                found_ids.add(skymap.findTract(sp).tract_id)
            except Exception:
                pass
    return sorted(found_ids)


tract_ids = find_tracts_for_coord(skymap, RA_CENTER, DEC_CENTER, radius_deg=TRACT_SEARCH_RADIUS_DEG)
print(f"Tracts covering {SELECTED_DDF}: {tract_ids}")

---
## 5. Schema Discovery (one-time probe)

### 5.1 Dia-Object Schema

#### Probe DIA_OBJ_OBJ 

In [ ]:
refs_probe = list(
    registry.queryDatasets(
        DIA_OBJ_TYPE,
        collections=COLLECTION,
        dataId={"skymap": SKYMAP_NAME, "tract": tract_ids[0]},
    )
)
assert refs_probe, "No dia_object ref found."

obj_raw = butler.get(refs_probe[0])
df_probe = obj_raw.to_pandas() if hasattr(obj_raw, "to_pandas") else obj_raw

print(f"index.name  : {df_probe.index.name!r}")
print(f"index.dtype : {df_probe.index.dtype}")
print(f"index[:3]   : {df_probe.index[:3].tolist()}")
print(f"columns     : {list(df_probe.columns)[:50]}")

#### Probe DIA-OBJECT ID name

In [ ]:
# Determine ID column
if df_probe.index.name is not None:
    OBJ_ID_COL = df_probe.index.name
    ID_IN_INDEX = True
    print(f"Object ID in index: '{OBJ_ID_COL}'")
elif any(c in df_probe.columns for c in ["diaObjectId", "dia_object_id"]):
    OBJ_ID_COL = next(c for c in ["diaObjectId", "dia_object_id"] if c in df_probe.columns)
    ID_IN_INDEX = False
    print(f"Object ID as column: '{OBJ_ID_COL}'")
else:
    OBJ_ID_COL = "row_id"
    ID_IN_INDEX = False
    print("WARNING: no ID found, using row index.")

#### Probe existence of FORCED_DIA_TYPE

In [ ]:
# Also probe dia_object_forced_source schema (patch-level)
refs_forced_probe = list(
    registry.queryDatasets(
        FORCED_DIA_TYPE,
        collections=COLLECTION,
        dataId={"skymap": SKYMAP_NAME, "tract": tract_ids[0]},
    )
)
print(f"dia_object_forced_source refs for tract {tract_ids[0]}: {len(refs_forced_probe)}")

if refs_forced_probe:
    frc_raw = butler.get(refs_forced_probe[0])
    df_fprobe = frc_raw.to_pandas() if hasattr(frc_raw, "to_pandas") else frc_raw
    print(f"index.name  : {df_fprobe.index.name!r}")
    print(f"columns     : {list(df_fprobe.columns)}")
    print(f"shape       : {df_fprobe.shape}")

In [ ]:
list(df_fprobe.columns)

---
## 6. Helper: `to_dataframe()`

In [ ]:
def to_dataframe(obj, id_col=None, id_in_index=None):
    """
    Convert Butler catalog to pandas DataFrame.
    If id_in_index is True, promotes the named index to a regular column.
    Falls back to global OBJ_ID_COL / ID_IN_INDEX if not provided.
    """
    _id_col = id_col if id_col is not None else OBJ_ID_COL
    _id_idx = id_in_index if id_in_index is not None else ID_IN_INDEX

    if isinstance(obj, pd.DataFrame):
        df = obj
    elif hasattr(obj, "to_pandas"):
        df = obj.to_pandas()
    elif hasattr(obj, "to_table"):
        df = obj.to_table().to_pandas()
    elif isinstance(obj, Table):
        df = obj.to_pandas()
    else:
        raise TypeError(f"Cannot convert {type(obj)} to DataFrame.")

    if _id_idx and _id_col not in df.columns:
        df = df.copy()
        df.insert(0, _id_col, df.index)
        df = df.reset_index(drop=True)
    elif _id_col == "row_id" and _id_col not in df.columns:
        df.insert(0, _id_col, range(len(df)))

    return df

---
## 7. Load DiaObject Catalog

### 7.1 Select only one TRACT

- In fact there is less Dia Object thus keep all tracts

In [ ]:
# tract_ids = [SELECTED_TRACT]

### 7.2 Retrieve Objects from the Butler

In [ ]:
dia_obj_frames = []

# loop on tracts
for tract_id in tract_ids:
    refs = list(
        registry.queryDatasets(
            DIA_OBJ_TYPE, collections=COLLECTION, dataId={"skymap": SKYMAP_NAME, "tract": tract_id}
        )
    )
    for ref in refs:
        try:
            df_tmp = to_dataframe(butler.get(ref))
            df_tmp["_tract"] = tract_id
            dia_obj_frames.append(df_tmp)
            print(f"  Tract {tract_id:6d} — {len(df_tmp):,} DiaObjects")
        except Exception as exc:
            print(f"  Tract {tract_id:6d} — ERROR: {exc}")

assert dia_obj_frames, "No dia_object data loaded."
df_dia_obj_all = pd.concat(dia_obj_frames, ignore_index=True)
n_before = len(df_dia_obj_all)
df_dia_obj_all = df_dia_obj_all.drop_duplicates(subset=OBJ_ID_COL)
gc.collect()
n_after = len(df_dia_obj_all)
print(f"\nTotal: {n_before:,} → {n_after:,} after dedup")
gc.collect()

### 7.3 Detect column name in object Table

In [ ]:
# Detect column names
RA_COL, DEC_COL = next(
    (
        (ra, dec)
        for ra, dec in [("ra", "dec"), ("coord_ra", "coord_dec"), ("RA", "DEC")]
        if ra in df_dia_obj_all.columns
    ),
    (None, None),
)
assert RA_COL, f"No RA/Dec found. Columns: {list(df_dia_obj_all.columns)}"

NSRC_COL = next((c for c in ["nDiaSources", "n_dia_sources"] if c in df_dia_obj_all.columns), None)

target_order = ["u", "g", "r", "i", "z", "y"]
BANDS = sorted(
    {
        c.split("_")[0]
        for c in df_dia_obj_all.columns
        if c.endswith("_psfFluxNdata") and len(c.split("_")[0]) == 1
    },
    key=lambda b: target_order.index(b) if b in target_order else 99,
)
BAND_COLORS = {"u": "purple", "g": "green", "r": "red", "i": "darkorange", "z": "sienna", "y": "black"}

print(f"ID   : '{OBJ_ID_COL}'")
print(f"RA   : '{RA_COL}'  Dec: '{DEC_COL}'")
print(f"nSrc : '{NSRC_COL}'")
print(f"Bands: {BANDS}")

### 7.4 Select object in a cone

In [ ]:
# Spatial cone cut
center_sky = SkyCoord(ra=RA_CENTER * u.deg, dec=DEC_CENTER * u.deg)
sep_deg = center_sky.separation(
    SkyCoord(ra=df_dia_obj_all[RA_COL].values * u.deg, dec=df_dia_obj_all[DEC_COL].values * u.deg)
).deg

df_dia_obj_cone = df_dia_obj_all[sep_deg <= CONE_RADIUS_DEG].copy()
df_dia_obj_cone["_sep_deg"] = sep_deg[sep_deg <= CONE_RADIUS_DEG]
print(f"In cone: {len(df_dia_obj_cone):,} / {len(df_dia_obj_all):,}")

In [ ]:
del df_dia_obj_all
gc.collect()

### 7.4 Select dia-object with enough sources

In [ ]:
# Filter on nDiaSources
if NSRC_COL:
    df_dia_obj_rich = (
        df_dia_obj_cone[df_dia_obj_cone[NSRC_COL] >= MIN_NSOURCES]
        .sort_values(NSRC_COL, ascending=False)
        .reset_index(drop=True)
    )
    print(
        f"Filtered: {len(df_dia_obj_rich):,}  "
        f"(max={df_dia_obj_rich[NSRC_COL].max()}, "
        f"med={df_dia_obj_rich[NSRC_COL].median():.1f})"
    )
else:
    df_dia_obj_rich = df_dia_obj_cone.copy()
    print("No NSRC_COL — keeping all.")

peek_cols = [
    c for c in [OBJ_ID_COL, RA_COL, DEC_COL, NSRC_COL, "_tract", "_sep_deg"] if c in df_dia_obj_rich.columns
]
display(df_dia_obj_rich[peek_cols].head(10))

---
## 8. Diagnostic Plots — DiaObject

### 8.1 Plot distribution of number of sources

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
if NSRC_COL:
    ax.hist(
        df_dia_obj_cone[NSRC_COL].dropna(),
        bins=100,
        log=True,
        color="steelblue",
        alpha=0.8,
        edgecolor="white",
        linewidth=0.3,
    )
    ax.axvline(MIN_NSOURCES, color="red", ls="--", lw=1.5, label=f"cut >= {MIN_NSOURCES}")
ax.set(
    xlabel="nDiaSources",
    ylabel="N DiaObjects",
    title=f"{SELECTED_DDF} — All in cone (N={len(df_dia_obj_cone):,})",
)
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)

ax = axes[1]
if NSRC_COL and len(df_dia_obj_rich) > 0:
    ax.hist(
        df_dia_obj_rich[NSRC_COL].dropna(),
        bins=50,
        color="darkorange",
        alpha=0.8,
        edgecolor="white",
        linewidth=0.3,
    )
    ax.set_title(f"{SELECTED_DDF} — Filtered (N={len(df_dia_obj_rich):,})")
ax.set(xlabel="nDiaSources", ylabel="N DiaObjects")
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f"{DIR_FIGS}/DiaObj_nSources_{SELECTED_DDF}.png", dpi=150, bbox_inches="tight")
plt.show()

### 8.2 Plot (Ra,Dec) distribution

In [ ]:
fig, ax = plt.subplots(figsize=(9, 8))
ax.set_facecolor("#f5f5f5")
ax.invert_xaxis()

ax.scatter(
    df_dia_obj_cone[RA_COL],
    df_dia_obj_cone[DEC_COL],
    s=1,
    color="lightgrey",
    alpha=0.4,
    zorder=1,
    label=f"All in cone ({len(df_dia_obj_cone):,})",
)

if NSRC_COL and len(df_dia_obj_rich) > 0:
    sc = ax.scatter(
        df_dia_obj_rich[RA_COL],
        df_dia_obj_rich[DEC_COL],
        c=df_dia_obj_rich[NSRC_COL],
        cmap="plasma",
        s=14,
        alpha=0.9,
        zorder=3,
        norm=mcolors.LogNorm(vmin=df_dia_obj_rich[NSRC_COL].min(), vmax=df_dia_obj_rich[NSRC_COL].max()),
        label=f"nSrc >= {MIN_NSOURCES} ({len(df_dia_obj_rich):,})",
    )
    plt.colorbar(sc, ax=ax).set_label("nDiaSources (log)", fontsize=10)

ax.plot(
    RA_CENTER,
    DEC_CENTER,
    marker="*",
    ms=18,
    color="gold",
    mec="black",
    mew=1,
    zorder=10,
    ls="none",
    label=f"{SELECTED_DDF} center",
)
cos_dec = np.cos(np.deg2rad(DEC_CENTER))
theta = np.linspace(0, 2 * np.pi, 361)
ax.plot(
    RA_CENTER + CONE_RADIUS_DEG / cos_dec * np.cos(theta),
    DEC_CENTER + CONE_RADIUS_DEG * np.sin(theta),
    "r--",
    lw=1.2,
    zorder=4,
    label=f"Cone {CONE_RADIUS_DEG} deg",
)

ax.set(xlabel="RA (deg)", ylabel="Dec (deg)", title=f"DP2 DiaObjects — {SELECTED_DDF}  (tracts {tract_ids})")
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3, ls="--")
plt.tight_layout()
plt.savefig(f"{DIR_FIGS}/DiaObj_sky_{SELECTED_DDF}.png", dpi=150, bbox_inches="tight")
plt.show()

### 8.3 Plot Dia-Flux error vs Dia-Flux

In [ ]:
# --- Paramètres de filtrage ---
N_SIGMA_IQR = 10.0  # Largeur du range (équivalent sigma)
TOP_N = min(2000, len(df_dia_obj_rich))
df_top = df_dia_obj_rich.head(TOP_N)

# 1. Collecte de toutes les données valides pour calculer un range GLOBAL
all_x, all_y = [], []
for band in BANDS:
    mc, sc = f"{band}_psfFluxMean", f"{band}_psfFluxSigma"
    if mc in df_top.columns and sc in df_top.columns:
        valid = df_top[mc].notna() & df_top[sc].notna()
        all_x.extend(df_top.loc[valid, mc])
        all_y.extend(df_top.loc[valid, sc])


# 2. Fonction pour calculer les limites basées sur l'IQR
def get_iqr_range(data, n_sigma):
    if len(data) == 0:
        return (0, 1)
    q1, q3 = np.percentile(data, [25, 75])
    iqr = q3 - q1
    median = np.median(data)
    # On convertit l'IQR en "sigma équivalent" (1 sigma ~ IQR/1.349)
    std_equivalent = iqr / 1.349
    return (median - n_sigma * std_equivalent, median + n_sigma * std_equivalent)


# Calcul des limites communes
x_range = get_iqr_range(all_x, N_SIGMA_IQR)
y_range = get_iqr_range(all_y, N_SIGMA_IQR)

# --- Visualisation ---
ncols = 3
nrows = int(np.ceil(len(BANDS) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 5 * nrows), constrained_layout=True)
axes = np.array(axes).flatten()

for idx, band in enumerate(BANDS):
    ax = axes[idx]
    mc, sc, nc = f"{band}_psfFluxMean", f"{band}_psfFluxSigma", f"{band}_psfFluxNdata"

    if mc not in df_top.columns:
        ax.set_visible(False)
        continue

    valid = df_top[mc].notna() & (df_top[nc] > 0)
    x = df_top.loc[valid, mc]
    y = df_top.loc[valid, sc] if sc in df_top.columns else np.zeros(valid.sum())
    c = df_top.loc[valid, nc] if nc in df_top.columns else "grey"

    sc_ = ax.scatter(x, y, c=c, cmap="viridis", s=10, alpha=0.6)
    plt.colorbar(sc_, ax=ax, label="nEpochs")

    # Application des limites identiques pour tous
    ax.set_xlim(x_range)
    ax.set_ylim(0, y_range[1])  # Souvent Sigma commence à 0, donc on fixe le min à 0

    ax.set(
        xlabel=f"Mean (nJy)",
        ylabel=f"Sigma (nJy)",
        title=f"Band {band} (N={valid.sum():,})",
    )
    ax.title.set_color(BAND_COLORS.get(band, "black"))
    ax.grid(True, alpha=0.3, linestyle="--")

# Nettoyage des axes vides
for idx in range(len(BANDS), len(axes)):
    axes[idx].set_visible(False)

fig.suptitle(
    f"{SELECTED_DDF} — Flux vs Variability (IQR Clipping: {N_SIGMA_IQR}σ eq.)", fontsize=16, fontweight="bold"
)

plt.savefig(f"{DIR_FIGS}/DiaObj_flux_summary_{SELECTED_DDF}.png", dpi=150, bbox_inches="tight")
plt.show()

### 8.4 Plot Histogram on the Number of source

In [ ]:
# Epochs per band
fig, ax = plt.subplots(figsize=(6, 5))
for band in BANDS:
    col = f"{band}_psfFluxNdata"
    if col in df_dia_obj_rich.columns:
        vals = df_dia_obj_rich[col].dropna()
        ax.hist(
            vals,
            bins=50,
            alpha=0.55,
            histtype="stepfilled",
            color=BAND_COLORS.get(band, "grey"),
            label=f"{band} (med={vals.median():.0f})",
        )
ax.set(
    xlabel="psfFluxNdata (epochs per band)",
    ylabel="N DiaObjects",
    title=f"{SELECTED_DDF} — Epochs per band  (N={len(df_dia_obj_rich):,})",
)
ax.legend(title="Band", fontsize=12)
ax.grid(True, alpha=0.3)
ax.set_yscale("log")
plt.tight_layout()
plt.savefig(f"{DIR_FIGS}/DiaObj_ndata_band_{SELECTED_DDF}.png", dpi=150, bbox_inches="tight")
plt.show()

---
## 9. Load DiaSources

In [ ]:
rich_ids = set(df_dia_obj_rich[OBJ_ID_COL].values)
print(f"Object IDs to keep: {len(rich_ids):,}")

#### Clean memory from dia objects

In [ ]:
# del df_dia_obj_rich
# gc.collect()

#### Get dia sources

In [ ]:
dia_src_frames = []
for tract_id in tract_ids:
    refs = list(
        registry.queryDatasets(
            DIA_SRC_TYPE, collections=COLLECTION, dataId={"skymap": SKYMAP_NAME, "tract": tract_id}
        )
    )
    for ref in refs:
        try:
            df_tmp = to_dataframe(butler.get(ref))
            join_col = next(
                (c for c in [OBJ_ID_COL, "diaObjectId", "dia_object_id"] if c in df_tmp.columns), None
            )
            if join_col:
                df_tmp = df_tmp[df_tmp[join_col].isin(rich_ids)]
            else:
                print(
                    f"  WARNING tract {tract_id}: join col not found. " f"Cols: {list(df_tmp.columns[:10])}"
                )
            dia_src_frames.append(df_tmp)
            print(f"  Tract {tract_id:6d} — {len(df_tmp):,} DiaSources")
        except Exception as exc:
            print(f"  Tract {tract_id:6d} — ERROR: {exc}")

if dia_src_frames:
    df_dia_src_rich = pd.concat(dia_src_frames, ignore_index=True)
    src_id = next((c for c in ["diaSourceId", "dia_source_id"] if c in df_dia_src_rich.columns), None)
    if src_id:
        df_dia_src_rich = df_dia_src_rich.drop_duplicates(subset=src_id)
    print(f"\nTotal DiaSources: {len(df_dia_src_rich):,}")
    print(f"Columns: {list(df_dia_src_rich.columns)}")
else:
    print("No DiaSources loaded.")
    df_dia_src_rich = pd.DataFrame()

In [ ]:
if len(df_dia_src_rich) > 0:
    MJD_COL = next(
        (c for c in ["midPointMjdTai", "midpointMjdTai", "mjd"] if c in df_dia_src_rich.columns), None
    )
    BAND_COL = next((c for c in ["band", "filterName", "filter"] if c in df_dia_src_rich.columns), None)
    FLUX_COL = next((c for c in ["psfFlux", "psf_flux"] if c in df_dia_src_rich.columns), None)
    FERR_COL = next((c for c in ["psfFluxErr", "psf_flux_err"] if c in df_dia_src_rich.columns), None)
    print(f"MJD={MJD_COL}  band={BAND_COL}  flux={FLUX_COL}+/-{FERR_COL}")
else:
    MJD_COL = BAND_COL = FLUX_COL = FERR_COL = None

### 9.1 Diagnostic show all measurements

In [ ]:
if MJD_COL and FLUX_COL and BAND_COL and len(df_dia_src_rich) > 0:
    fig, ax = plt.subplots(figsize=(16, 6))
    for band, grp in df_dia_src_rich.groupby(BAND_COL):
        ax.scatter(
            grp[MJD_COL],
            grp[FLUX_COL],
            s=1,
            alpha=0.2,
            color=BAND_COLORS.get(str(band), "grey"),
            label=str(band),
        )
    ax.set(
        xlabel="MJD (TAI)",
        ylabel="PSF flux (nJy)",
        title=f"DiaSources — {SELECTED_DDF} ({len(df_dia_src_rich):,})",
    )
    ax.legend(title="Band", fontsize=12, markerscale=6)
    ax.grid(True, alpha=0.3)
    add_date_top_axis(ax, n_ticks=8)
    plt.tight_layout()
    plt.savefig(f"{DIR_FIGS}/DiaSrc_scatter_{SELECTED_DDF}.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("DiaSources scatter skipped.")

In [ ]:
del df_dia_src_rich
gc.collect()

---
## 10. Load Forced Photometry (`dia_object_forced_source`)

This dataset has dimensions `(skymap, tract, patch)` — there are **~100 patch refs per tract**.
Each patch file contains per-visit forced-PSF photometry for the DiaObjects in that patch.
We iterate over all patch refs, load, filter on `rich_ids`, and concatenate.

**NOTE :** `dia_object_forced_source` does not contain any time column.
It only carries columns like ('diaObjectId', 'parentObjectId', 'coord_ra', 'coord_dec', 'visit', 'detector', 'band', ...).

**Solution :** the `visitTable` retrieved from the registry in Section 2.2 maps each `visit id` (column `id`)
to its observation time (column `mjd`).  After loading all forced-photometry rows we merge this lookup
table on the `visit` column, creating a new column **`mjd_visit`** that is used as the time axis for light curves.

### 10.1 Load forced photometry

In [ ]:
forced_frames = []

# loop on tracts
for tract_id in tract_ids:
    # Query all patch refs for this tract (no patch constraint -> all patches)
    refs = list(
        registry.queryDatasets(
            FORCED_DIA_TYPE,
            collections=COLLECTION,
            dataId={"skymap": SKYMAP_NAME, "tract": tract_id},
        )
    )
    print(f"Tract {tract_id:6d} — {len(refs)} patch refs")

    # loop on force photometry dia sources refs for that tract
    for idx, ref in enumerate(refs):
        patch_id = ref.dataId.get("patch", "?")
        try:
            obj = butler.get(ref)
            df_tmp = obj.to_pandas() if hasattr(obj, "to_pandas") else obj

            # Detect join column in this table
            join_col = next(
                (c for c in [OBJ_ID_COL, "diaObjectId", "dia_object_id"] if c in df_tmp.columns), None
            )
            # If ID is in index (same convention as dia_object)
            if join_col is None and df_tmp.index.name in [OBJ_ID_COL, "diaObjectId", "dia_object_id"]:
                df_tmp = df_tmp.copy()
                df_tmp.insert(0, df_tmp.index.name, df_tmp.index)
                df_tmp = df_tmp.reset_index(drop=True)
                join_col = df_tmp.columns[0]

            if join_col:
                df_filt = df_tmp[df_tmp[join_col].isin(rich_ids)]
            else:
                print(
                    f"    Patch {patch_id}: no join col. "
                    f"idx={df_tmp.index.name!r}  cols={list(df_tmp.columns[:8])}"
                )
                df_filt = df_tmp  # keep all — will be filtered later

            if len(df_filt) > 0:
                df_filt = df_filt.copy()
                df_filt["_tract"] = tract_id
                df_filt["_patch"] = patch_id
                forced_frames.append(df_filt)

        except Exception as exc:
            print(f"    Patch {patch_id} ERROR: {exc}")

if forced_frames:
    df_forced_rich = pd.concat(forced_frames, ignore_index=True)
    print(f"\nTotal forced rows: {len(df_forced_rich):,}")
    print(f"All columns ({len(df_forced_rich.columns)}):")
    print(list(df_forced_rich.columns))
else:
    print("No forced photometry loaded.")
    df_forced_rich = pd.DataFrame()

### 10.2 Load visitTable

In [ ]:
FULLFILENAME_VISITS = os.path.join(OUTPUT_DIR, FILENAME_VISITS)
print(f"Visit Filename : {FULLFILENAME_VISITS} ")

In [ ]:
df_visitTable = load_visits_file(FULLFILENAME_VISITS)

### 10.3 Associate Visit and MJD

In [ ]:
# =========================================================================
# Join visit MJD into df_forced_rich
# =========================================================================
# dia_object_forced_source carries a 'visit' column (integer visit id) but NO
# time information.  We therefore build a small lookup table from visitTable
# (loaded in Section 2.2) and merge it on the 'visit' column.
#
# visitTable has columns:
#   id   -> visit identifier (same as the 'visit' column in df_forced_rich)
#   mjd  -> Modified Julian Date of the exposure start (UTC)
#
# After the merge, df_forced_rich gains:
#   'mjd_visit'  -> float MJD of the observation
#   'date_visit' -> ISO date string YYYY-MM-DD for human-readable display
# =========================================================================

if len(df_forced_rich) > 0 and "visit" in df_forced_rich.columns:
    # Build a minimal lookup: visit_id -> mjd
    visit_mjd_lookup = df_visitTable[["id", "mjd"]].copy()
    visit_mjd_lookup = visit_mjd_lookup.rename(columns={"id": "visit", "mjd": "mjd_visit"})
    visit_mjd_lookup = visit_mjd_lookup.drop_duplicates(subset="visit")

    # Left join: keep all forced rows; unmatched visits get NaN
    df_forced_rich = df_forced_rich.merge(visit_mjd_lookup, on="visit", how="left")

    # Add human-readable ISO date column for convenience
    df_forced_rich["date_visit"] = pd.to_datetime(
        df_forced_rich["mjd_visit"] + 2400000.5, unit="D", origin="julian"
    ).dt.strftime("%Y-%m-%d")

    n_matched = df_forced_rich["mjd_visit"].notna().sum()
    n_total = len(df_forced_rich)
    print(f"mjd_visit joined: {n_matched:,} / {n_total:,} rows matched a visit time")
    if n_matched < n_total:
        # Some visit ids in forced photometry were not found in visitTable.
        # This can happen if the visitTable query (WHERE_CLAUSE_DATE) does not
        # cover the full date range of the collection. Lower DATE_START if needed.
        missing_visits = df_forced_rich.loc[df_forced_rich["mjd_visit"].isna(), "visit"].unique()
        print(f"  WARNING: {n_total - n_matched} rows have no MJD match.")
        print(f"  First missing visit ids: {missing_visits[:5]}")

    print(
        f"mjd_visit range : {df_forced_rich['mjd_visit'].min():.2f} "
        f"-> {df_forced_rich['mjd_visit'].max():.2f}"
    )
    print(
        f"date_visit range: {df_forced_rich['date_visit'].min()} " f"-> {df_forced_rich['date_visit'].max()}"
    )
else:
    print("Skipping MJD join: df_forced_rich is empty or 'visit' column not found.")

### 10.3  Check availability of all required variables for LC

In [ ]:
# Check names of columns in force photometry table
# NOTE: 'mjd_visit' is the column added by the MJD join cell above
if len(df_forced_rich) > 0:
    # 'mjd_visit' is the column we created by joining visitTable
    FMJD_COL = next(
        (c for c in ["mjd_visit", "midpointMjdTai", "midPointMjdTai", "mjd"] if c in df_forced_rich.columns),
        None,
    )
    FBAND_COL = next((c for c in ["band", "filterName", "filter"] if c in df_forced_rich.columns), None)
    FFLUX_COL = next((c for c in ["psfFlux", "psf_flux"] if c in df_forced_rich.columns), None)
    FFERR_COL = next((c for c in ["psfFluxErr", "psf_flux_err"] if c in df_forced_rich.columns), None)
    FDIFF_COL = next((c for c in ["psfDiffFlux", "psf_diff_flux"] if c in df_forced_rich.columns), None)
    FDERR_COL = next(
        (c for c in ["psfDiffFluxErr", "psf_diff_flux_err"] if c in df_forced_rich.columns), None
    )
    FOBJ_COL = next(
        (c for c in [OBJ_ID_COL, "diaObjectId", "dia_object_id"] if c in df_forced_rich.columns), None
    )

    print(f"MJD  : {FMJD_COL}")
    print(f"Band : {FBAND_COL}")
    print(f"Flux : {FFLUX_COL} +/- {FFERR_COL}")
    print(f"Diff : {FDIFF_COL} +/- {FDERR_COL}")
    print(f"ObjID: {FOBJ_COL}")
else:
    FMJD_COL = FBAND_COL = FFLUX_COL = FFERR_COL = None
    FDIFF_COL = FDERR_COL = FOBJ_COL = None

### 10.4  Plot Forced LC of the most-detected DiaObject : Use science flux and diff flux

In [ ]:
def plot_dia_object_lightcurves(dia_id, df_forced, df_dia_obj, selected_ddf, save_dir):
    """
    Plot forced and difference flux light curves for a specific DiaObject.

    Parameters:
    dia_id : int or str
        The unique ID of the DiaObject (objectId).
    df_forced : pd.DataFrame
        The forced photometry dataframe containing flux measurements.
    df_dia_obj : pd.DataFrame
        The DiaObject summary table to retrieve metadata (like nDiaSrc).
    selected_ddf : str
        Name of the DDF for labeling and file naming.
    save_dir : str
        Directory path to save the generated figures.
    """

    # 1. Filter forced data for this specific object and sort by time
    df_lc = df_forced[df_forced[FOBJ_COL] == dia_id].sort_values(FMJD_COL)

    # 2. Retrieve metadata for the title
    # We look for the object in the rich summary table
    obj_info = df_dia_obj[df_dia_obj[OBJ_ID_COL] == dia_id]
    n_dia_src = obj_info.iloc[0][NSRC_COL] if not obj_info.empty and NSRC_COL else "N/A"

    if len(df_lc) == 0:
        print(f"Skipping {dia_id}: No forced photometry rows found.")
        return

    # 3. Determine number of subplots (Forced PSF Flux and/or Difference PSF Flux)
    show_diff = FDIFF_COL and FDIFF_COL in df_lc.columns
    nrows_fig = 2 if show_diff else 1

    fig, axes = plt.subplots(nrows_fig, 1, figsize=(16, 3 * nrows_fig), sharex=True)
    if nrows_fig == 1:
        axes = [axes]

    # --- TOP PLOT: Science / Forced PSF Flux ---
    ax0 = axes[0]
    for band, grp in df_lc.groupby(FBAND_COL):
        yerr = grp[FFERR_COL].values if FFERR_COL else None
        ax0.errorbar(
            grp[FMJD_COL],
            grp[FFLUX_COL],
            yerr=yerr,
            fmt="o",
            ms=4,
            alpha=0.8,
            capsize=2,
            color=BAND_COLORS.get(str(band), "grey"),
            label=f"band {band}",
        )

    ax0.axhline(0, color="black", lw=1, ls="--", alpha=0.5)
    ax0.set_ylabel("Science PSF flux (nJy)", fontsize=12)
    ax0.set_title(
        f"DiaObject: {dia_id} | nDiaSrc: {n_dia_src} | [{selected_ddf}]",
        fontsize=15,
        fontweight="bold",
        pad=25,
    )
    ax0.legend(loc="upper left", bbox_to_anchor=(1, 1), title="Bands")
    ax0.grid(True, alpha=0.3)

    # Add the date axis on top (Human readable YYYY-MM-DD)
    add_date_top_axis(ax0, n_ticks=8)

    # --- BOTTOM PLOT: Difference PSF Flux ---
    if show_diff:
        ax1 = axes[1]
        for band, grp in df_lc.groupby(FBAND_COL):
            yerr = grp[FDERR_COL].values if FDERR_COL else None
            ax1.errorbar(
                grp[FMJD_COL],
                grp[FDIFF_COL],
                yerr=yerr,
                fmt="o",
                ms=4,
                alpha=0.8,
                capsize=2,
                color=BAND_COLORS.get(str(band), "grey"),
                label=f"band {band}",
            )
        ax1.axhline(0, color="black", lw=1, ls="--", alpha=0.5)
        ax1.set_ylabel("Diff PSF flux (nJy)", fontsize=12)
        ax1.grid(True, alpha=0.3)
        ax1.legend(loc="upper left", bbox_to_anchor=(1, 1), title="Bands")

    # Common X-axis settings
    axes[-1].set_xlabel("MJD (Modified Julian Date)", fontsize=13, labelpad=10)

    # 4. Save and show
    plt.tight_layout()
    filename = f"ForcedLC_{dia_id}_{selected_ddf}.png"
    plt.savefig(f"{save_dir}/{filename}", dpi=150, bbox_inches="tight")
    plt.show()

In [ ]:
# =========================================================================
# Execution: Loop over the Top N objects
# =========================================================================
NUM_OBJECTS_TO_PLOT = 20  # Set how many objects you want to inspect

if len(df_dia_obj_rich) > 0:
    # Get the list of IDs for the top N objects (e.g., most detected)
    top_ids = df_dia_obj_rich.head(NUM_OBJECTS_TO_PLOT)[OBJ_ID_COL].values

    print(f"Generating Light Curves for the top {len(top_ids)} objects...")

    for current_id in top_ids:
        plot_dia_object_lightcurves(
            dia_id=current_id,
            df_forced=df_forced_rich,
            df_dia_obj=df_dia_obj_rich,
            selected_ddf=SELECTED_DDF,
            save_dir=DIR_FIGS,
        )
else:
    print("No DiaObjects found to plot.")

### 10.5  Plot Forced LC of the most-detected DiaObject : Use mag(science flux) only

C'est une question fondamentale en photométrie de différence (DIA). Comme le flux de différence ($f_{diff}$) est le résultat de la soustraction d'un modèle (le template) à une image directe, il est tout à fait normal — et même statistiquement nécessaire — qu'il soit parfois négatif à cause du bruit de fond du ciel, surtout là où il n'y a pas de source variable réelle.Le problème est que la formule standard de la magnitude AB :$$m_{AB} = -2.5 \log_{10}(f_{nJy}) + 31.4$$ne supporte pas les valeurs négatives ($\log(x)$ n'est pas défini pour $x \le 0$).

In [ ]:
def plot_dia_object_lightcurves_mag(dia_id, df_forced, df_dia_obj, selected_ddf, save_dir):
    """
    Plot forced light curves in AB Magnitudes for a specific DiaObject.
    Negative flux values are handled by being ignored (NaN in magnitudes).
    """

    # 1. Create df_lc by filtering the forced photometry table
    # This was the missing line causing the NameError!
    df_lc = df_forced[df_forced[FOBJ_COL] == dia_id].sort_values(FMJD_COL).copy()

    if len(df_lc) == 0:
        print(f"Skipping {dia_id}: No forced photometry rows found.")
        return

    # 2. Retrieve metadata
    obj_info = df_dia_obj[df_dia_obj[OBJ_ID_COL] == dia_id]
    n_dia_src = obj_info.iloc[0][NSRC_COL] if not obj_info.empty and NSRC_COL else "N/A"

    # 3. Calculate Magnitude AB: m = -2.5 * log10(flux_nJy) + 31.4
    # We use np.where to handle non-positive flux safely
    df_lc["mag_psf"] = np.where(
        df_lc[FFLUX_COL] > 0, -2.5 * np.log10(df_lc[FFLUX_COL].replace(0, np.nan)) + 31.4, np.nan
    )

    # 4. Error propagation: sigma_mag = 1.0857 * (sigma_flux / flux)
    # Only calculate where flux is positive to avoid inf/nan issues
    df_lc["mag_err"] = np.where(df_lc[FFLUX_COL] > 0, 1.0857 * (df_lc[FFERR_COL] / df_lc[FFLUX_COL]), np.nan)

    # 5. Plotting
    fig, ax = plt.subplots(figsize=(16, 4.0))

    for band, grp in df_lc.groupby(FBAND_COL):
        # We only plot points with a valid magnitude
        valid = grp["mag_psf"].notna()
        if valid.any():
            ax.errorbar(
                grp.loc[valid, FMJD_COL],
                grp.loc[valid, "mag_psf"],
                yerr=grp.loc[valid, "mag_err"],
                fmt="o",
                ms=5,
                alpha=0.8,
                capsize=2,
                color=BAND_COLORS.get(str(band), "grey"),
                label=f"band {band}",
            )

    # IMPORTANT: Invert Y-axis for magnitudes (lower magnitude = brighter)
    ax.invert_yaxis()

    # Aesthetics
    ax.set_ylabel("Science \n PSF-Magnitude (AB)", fontsize=12, fontweight="bold")
    ax.set_xlabel("MJD (Modified Julian Date)", fontsize=12, fontweight="bold")
    ax.set_title(
        f"Forced Mag LC: {dia_id} | nDiaSrc: {n_dia_src} | [{selected_ddf}]",
        fontsize=15,
        fontweight="bold",
        pad=30,
    )

    ax.grid(True, alpha=0.3, linestyle="--")
    ax.legend(loc="upper left", bbox_to_anchor=(1, 1), title="Bands")

    # Add human-readable date on top
    add_date_top_axis(ax, n_ticks=8)

    # 6. Save and show
    plt.tight_layout()
    filename = f"ForcedMagLC_{dia_id}_{selected_ddf}.png"
    plt.savefig(f"{save_dir}/{filename}", dpi=150, bbox_inches="tight")
    plt.show()

In [ ]:
# =========================================================================
# Execution: Loop over the Top N objects
# =========================================================================
NUM_OBJECTS_TO_PLOT = 20  # Set how many objects you want to inspect

if len(df_dia_obj_rich) > 0:
    # Get the list of IDs for the top N objects (e.g., most detected)
    top_ids = df_dia_obj_rich.head(NUM_OBJECTS_TO_PLOT)[OBJ_ID_COL].values

    print(f"Generating Light Curves for the top {len(top_ids)} objects...")

    for current_id in top_ids:
        plot_dia_object_lightcurves_mag(
            dia_id=current_id,
            df_forced=df_forced_rich,
            df_dia_obj=df_dia_obj_rich,
            selected_ddf=SELECTED_DDF,
            save_dir=DIR_FIGS,
        )
else:
    print("No DiaObjects found to plot.")

### 10.7  Plot Forced LC of the most-detected DiaObject : Non linear transformation of the Flux

Voici les trois approches standards utilisées en astronomie pour régler ce problème :
- 1. La "Asinh Magnitude" (ou Luptitudes)C'est la méthode adoptée par le SDSS et préconisée pour LSST. On remplace le logarithme par un sinus hyperbolique inverse ($asinh$).
             - `Avantage :` Elle se comporte comme une magnitude normale pour les flux élevés, mais reste finie et définie pour un flux nul ou négatif.
             - `Comportement :` Près de zéro, elle devient linéaire avec le flux, ce qui permet de visualiser le bruit de manière symétrique.
- 2. Le "Flux à deux faces" (Sign-Preserving Log) - Pour garder une échelle logarithmique tout en voyant les points négatifs, on utilise souvent cette transformation pour l'affichage : $$y = \text{sign}(f) \cdot 2.5 \log_{10}(|f| + 1)$$Cela permet de voir les "trous" de flux (soustractions excessives) de la même manière que les pics.
     
- 3. Les Limites de Détection (Upper Limits)C'est l'approche la plus courante pour les publications :Si le flux est significatif (ex: $f > 3\sigma$), on trace la magnitude. Si le flux est négatif ou non significatif ($f < 3\sigma$), on trace une flèche vers le bas indiquant la limite de détection ($m_{lim} = -2.5 \log_{10}(3\sigma) + 31.4$).

#### Luptitudes

C'est une excellente approche. Les Luptitudes (ou magnitudes $asinh$) ont été inventées par Robert Lupton et ses collègues pour le SDSS précisément pour résoudre votre problème : avoir un comportement logarithmique pour les flux brillants et un comportement linéaire (qui accepte le zéro et le négatif) pour les flux faibles.Le concept mathématiqueAu lieu de $m = -2.5 \log_{10}(f)$, on utilise :$$\mu = a - \frac{2.5}{\ln(10)} \left[ \text{asinh}\left(\frac{f}{2b}\right) + \ln(b) \right]$$où $b$ est le "softening parameter" (paramètre d'adoucissement). Il est généralement fixé proche du niveau du bruit de fond ($\sigma_{sky}$).

Pourquoi c'est "magique" ?
 
- 1. **Plus de trous** : Contrairement au plot précédent, tous vos points (même ceux à $-50$ nJy) sont tracés. Vous voyez enfin la réalité statistique de votre bruit de fond.

- 2. **Linéarité près de zéro** : À bas flux, la distance entre les points est proportionnelle au flux (en nJy).

- 3. **Logarithmique à haut flux** : Pour vos points brillants, la valeur de la Luptitude sera identique à la magnitude AB classique.

- 4. **Erreurs constantes** : Les barres d'erreur ne "volent" plus dans tous les sens quand le flux approche de zéro ; elles restent proportionnelles au bruit de mesure.

In [ ]:
def plot_dia_object_luptitudes(dia_id, df_forced, df_dia_obj, selected_ddf, save_dir):
    """
    Plot light curves using asinh magnitudes (Luptitudes).
    This handles zero and negative fluxes gracefully while
    preserving a magnitude-like scale for bright sources.
    """

    # 1. Prepare data
    df_lc = df_forced[df_forced[FOBJ_COL] == dia_id].sort_values(FMJD_COL).copy()
    if len(df_lc) == 0:
        return

    # 2. Metadata
    obj_info = df_dia_obj[df_dia_obj[OBJ_ID_COL] == dia_id]
    n_dia_src = obj_info.iloc[0][NSRC_COL] if not obj_info.empty and NSRC_COL else "N/A"

    # 3. Luptitude calculation parameters
    # 'b' is the softening parameter. A common choice is the
    # median flux error of the object to represent the noise floor.
    b = df_lc[FFERR_COL].median()
    if b <= 0 or np.isnan(b):
        b = 1.0  # Fallback

    # Pre-factor 2.5 / ln(10) is approx 1.0857
    log10_e_inv = 1.085736

    # Asinh Mag formula: m = -1.0857 * (asinh(f / 2b) + ln(b)) + zero_point
    # For nJy to AB, the constant offset is related to 31.4
    # We calibrate the offset so it matches standard mag for high flux
    zero_point = 31.4

    df_lc["luptitude"] = -2.5 * log10_e_inv * (np.arcsinh(df_lc[FFLUX_COL] / (2 * b)) + np.log(b)) + (
        zero_point + 2.5 * np.log10(b)
    )

    # Error propagation for Luptitudes: sigma_mu = sigma_f / sqrt(f^2 + (2b)^2) * 1.0857
    df_lc["lup_err"] = log10_e_inv * (df_lc[FFERR_COL] / np.sqrt(df_lc[FFLUX_COL] ** 2 + (2 * b) ** 2))

    # 4. Plotting
    fig, ax = plt.subplots(figsize=(16, 4))

    for band, grp in df_lc.groupby(FBAND_COL):
        ax.errorbar(
            grp[FMJD_COL],
            grp["luptitude"],
            yerr=grp["lup_err"],
            fmt="o",
            ms=5,
            alpha=0.8,
            capsize=2,
            color=BAND_COLORS.get(str(band), "grey"),
            label=f"band {band}",
        )

    # Invert Y-axis as usual for magnitudes
    ax.invert_yaxis()

    ax.set_ylabel("Asinh Magnitude\n (Luptitude)", fontsize=12, fontweight="bold")
    ax.set_xlabel("MJD (Modified Julian Date)", fontsize=12, fontweight="bold")
    ax.set_title(
        f"Luptitude LC: {dia_id} | Softening b={b:.2f} nJy | [{selected_ddf}]",
        fontsize=15,
        fontweight="bold",
        pad=30,
    )

    ax.grid(True, alpha=0.3, linestyle="--")
    ax.legend(loc="upper left", bbox_to_anchor=(1, 1))

    add_date_top_axis(ax, n_ticks=8)

    plt.tight_layout()
    filename = f"LuptitudeLC_{dia_id}_{selected_ddf}.png"
    plt.savefig(f"{save_dir}/{filename}", dpi=150, bbox_inches="tight")
    plt.show()

In [ ]:
# =========================================================================
# Execution: Loop over the Top N objects
# =========================================================================
NUM_OBJECTS_TO_PLOT = 20  # Set how many objects you want to inspect

if len(df_dia_obj_rich) > 0:
    # Get the list of IDs for the top N objects (e.g., most detected)
    top_ids = df_dia_obj_rich.head(NUM_OBJECTS_TO_PLOT)[OBJ_ID_COL].values

    print(f"Generating Light Curves for the top {len(top_ids)} objects...")

    for current_id in top_ids:
        plot_dia_object_luptitudes(
            dia_id=current_id,
            df_forced=df_forced_rich,
            df_dia_obj=df_dia_obj_rich,
            selected_ddf=SELECTED_DDF,
            save_dir=DIR_FIGS,
        )
else:
    print("No DiaObjects found to plot.")

### 10.7  Plot Forced LC of the most-detected DiaObject : Dual Luptitude

Pourquoi c'est le "Combo" gagnant :Comparaison visuelle directe : 

- **Si vous voyez un sursaut dans le flux DIA mais que le flux Science reste plat**, c'est probablement un artefact ou une mauvaise soustraction.

- **Si les deux montent en même temps, la variabilité est réelle**.


- **Stabilité du bruit** : Sur le subplot du bas (Diff), les points qui "oscillent" autour de zéro ne disparaissent pas (comme avec le $\log$), ils forment une bande de points autour d'une magnitude de référence, ce qui permet de voir la qualité de la soustraction du template.


- **Inversion des axes** : Les deux axes Y sont inversés pour respecter la convention astronomique : plus le point est haut, plus la source est brillante.

-  C'est d'ailleurs la meilleure façon de diagnostiquer ce qui se passe sur un objet variable : le flux scientifique (Science PSF Flux) vous donne la luminosité totale (étoile + variabilité), tandis que le flux DIA (Diff PSF Flux) vous montre uniquement ce qui a changé par rapport au template.

- Utiliser les Luptitudes pour les deux subplots permet de garder une échelle cohérente en magnitudes, même pour le flux de différence qui est souvent proche de zéro ou négatif.

Voici le code combinant les deux subplots en mode Luptitudes :

In [ ]:
def plot_dia_object_dual_luptitudes(dia_id, df_forced, df_dia_obj, selected_ddf, save_dir):
    """
    Plot dual light curves (Science and Difference) using Luptitudes (asinh mag).
    Handles negative values gracefully for both photometry types.
    """
    import numpy as np

    # 1. Data Preparation
    df_lc = df_forced[df_forced[FOBJ_COL] == dia_id].sort_values(FMJD_COL).copy()
    if len(df_lc) == 0:
        return

    # 2. Metadata retrieval
    obj_info = df_dia_obj[df_dia_obj[OBJ_ID_COL] == dia_id]
    n_dia_src = obj_info.iloc[0][NSRC_COL] if not obj_info.empty and NSRC_COL else "N/A"

    # 3. Luptitude Calculation Engine
    def to_luptitude(flux, flux_err, zero_point=31.4):
        b = flux_err.median()
        if b <= 0 or np.isnan(b):
            b = 1.0
        log10_e_inv = 1.085736

        # asinh magnitude formula
        lup = -2.5 * log10_e_inv * (np.arcsinh(flux / (2 * b)) + np.log(b)) + (zero_point + 2.5 * np.log10(b))
        # error propagation
        lup_err = log10_e_inv * (flux_err / np.sqrt(flux**2 + (2 * b) ** 2))
        return lup, lup_err, b

    # Compute for Science Flux
    df_lc["lup_sci"], df_lc["lup_sci_err"], b_sci = to_luptitude(df_lc[FFLUX_COL], df_lc[FFERR_COL])

    # Compute for Difference Flux (if column exists)
    show_diff = FDIFF_COL in df_lc.columns
    if show_diff:
        df_lc["lup_diff"], df_lc["lup_diff_err"], b_diff = to_luptitude(df_lc[FDIFF_COL], df_lc[FDERR_COL])

    # 4. Plotting
    nrows = 2 if show_diff else 1
    fig, axes = plt.subplots(nrows, 1, figsize=(16, 3 * nrows), sharex=True)
    if nrows == 1:
        axes = [axes]

    # --- Top Plot: Science Luptitudes ---
    ax0 = axes[0]
    for band, grp in df_lc.groupby(FBAND_COL):
        ax0.errorbar(
            grp[FMJD_COL],
            grp["lup_sci"],
            yerr=grp["lup_sci_err"],
            fmt="o",
            ms=4,
            alpha=0.7,
            color=BAND_COLORS.get(str(band), "grey"),
            label=f"band {band}",
        )

    ax0.invert_yaxis()
    ax0.set_ylabel("Sci. Lupt.\n (asinh mag)", fontweight="bold")
    ax0.set_title(
        f"Dual Luptitude LC: {dia_id} | nDiaSrc: {n_dia_src} | [{selected_ddf}]",
        fontsize=14,
        fontweight="bold",
        pad=35,
    )
    ax0.legend(loc="upper left", bbox_to_anchor=(1, 1), title="Bands")
    ax0.grid(True, alpha=0.2)
    add_date_top_axis(ax0)

    # --- Bottom Plot: Difference Luptitudes ---
    if show_diff:
        ax1 = axes[1]
        for band, grp in df_lc.groupby(FBAND_COL):
            ax1.errorbar(
                grp[FMJD_COL],
                grp["lup_diff"],
                yerr=grp["lup_diff_err"],
                fmt="o",
                ms=4,
                alpha=0.7,
                color=BAND_COLORS.get(str(band), "grey"),
            )

        ax1.invert_yaxis()
        ax1.set_ylabel("Diff Lupt.\n (asinh mag)", fontweight="bold")
        ax1.grid(True, alpha=0.2)

    axes[-1].set_xlabel("MJD (Modified Julian Date)", fontsize=12)

    plt.tight_layout()
    plt.savefig(f"{save_dir}/DualLuptitude_{dia_id}.png", dpi=150, bbox_inches="tight")
    plt.show()

In [ ]:
def plot_dia_object_dual_luptitudes_refined(dia_id, df_forced, df_dia_obj, selected_ddf, save_dir):
    """
    Plot dual light curves with band-specific 5-sigma limiting magnitudes.
    """
    import numpy as np

    # 1. Data Prep
    df_lc = df_forced[df_forced[FOBJ_COL] == dia_id].sort_values(FMJD_COL).copy()
    if len(df_lc) == 0:
        return

    obj_info = df_dia_obj[df_dia_obj[OBJ_ID_COL] == dia_id]
    n_dia_src = obj_info.iloc[0][NSRC_COL] if not obj_info.empty and NSRC_COL else "N/A"

    # Helper function for Luptitude and 5-sigma limit
    def get_lup_stats(flux, flux_err, b_value, zero_point=31.4):
        log10_e_inv = 1.085736
        lup = -2.5 * log10_e_inv * (np.arcsinh(flux / (2 * b_value)) + np.log(b_value)) + (
            zero_point + 2.5 * np.log10(b_value)
        )
        lup_err = log10_e_inv * (flux_err / np.sqrt(flux**2 + (2 * b_value) ** 2))
        # 5-sigma is simply the luptitude of a flux = 5*b
        lup_5s = -2.5 * log10_e_inv * (np.arcsinh(5 * b_value / (2 * b_value)) + np.log(b_value)) + (
            zero_point + 2.5 * np.log10(b_value)
        )
        return lup, lup_err, lup_5s

    # 2. Plotting
    show_diff = FDIFF_COL in df_lc.columns
    nrows = 2 if show_diff else 1
    fig, axes = plt.subplots(nrows, 1, figsize=(16, 4 * nrows), sharex=True)
    if nrows == 1:
        axes = [axes]

    for band, grp in df_lc.groupby(FBAND_COL):
        color = BAND_COLORS.get(str(band), "grey")

        # --- SCIENCE PLOT (Top) ---
        b_sci = grp[FFERR_COL].median()
        if b_sci <= 0 or np.isnan(b_sci):
            b_sci = 1.0

        lup_s, err_s, lim_s = get_lup_stats(grp[FFLUX_COL], grp[FFERR_COL], b_sci)

        axes[0].errorbar(
            grp[FMJD_COL], lup_s, yerr=err_s, fmt="o", ms=4, alpha=0.7, color=color, label=f"band {band}"
        )
        axes[0].axhline(lim_s, color=color, linestyle=":", alpha=0.3, lw=1)

        # --- DIFFERENCE PLOT (Bottom) ---
        if show_diff:
            b_diff = grp[FDERR_COL].median()
            if b_diff <= 0 or np.isnan(b_diff):
                b_diff = 1.0

            lup_d, err_d, lim_d = get_lup_stats(grp[FDIFF_COL], grp[FDERR_COL], b_diff)

            axes[1].errorbar(grp[FMJD_COL], lup_d, yerr=err_d, fmt="o", ms=4, alpha=0.7, color=color)
            # This line is crucial to see which "flares" are real
            axes[1].axhline(lim_d, color=color, linestyle=":", alpha=0.3, lw=1)

    # 3. Aesthetics
    for ax in axes:
        ax.invert_yaxis()
        ax.grid(True, alpha=0.15)
        ax.legend(loc="upper left", bbox_to_anchor=(1, 1))

    axes[0].set_ylabel("Sci. Luptitude", fontweight="bold")
    axes[0].set_title(f"Object {dia_id} | nDiaSrc: {n_dia_src}", fontsize=14, fontweight="bold")

    if show_diff:
        axes[1].set_ylabel("Diff. Luptitude", fontweight="bold")

    axes[-1].set_xlabel("MJD")
    add_date_top_axis(axes[0])

    plt.tight_layout()
    plt.show()

In [ ]:
# =========================================================================
# Execution: Loop over the Top N objects
# =========================================================================
NUM_OBJECTS_TO_PLOT = 20  # Set how many objects you want to inspect

if len(df_dia_obj_rich) > 0:
    # Get the list of IDs for the top N objects (e.g., most detected)
    top_ids = df_dia_obj_rich.head(NUM_OBJECTS_TO_PLOT)[OBJ_ID_COL].values

    print(f"Generating Light Curves for the top {len(top_ids)} objects...")

    for current_id in top_ids:
        plot_dia_object_dual_luptitudes_refined(
            dia_id=current_id,
            df_forced=df_forced_rich,
            df_dia_obj=df_dia_obj_rich,
            selected_ddf=SELECTED_DDF,
            save_dir=DIR_FIGS,
        )
else:
    print("No DiaObjects found to plot.")

**Ce que cette ligne rouge vous indique :**
- Sur le plot Science : Elle montre la profondeur typique de vos images. Si vos points sont bien au-dessus de cette ligne (plus bas sur le graphique inversé), l'étoile est détectée avec un SNR élevé.
- Sur le plot Diff : C'est le plus intéressant. Comme le flux de différence est souvent proche de zéro, la plupart de vos points seront "en dessous" de la ligne rouge (donc plus hauts sur le plot). Seuls les points qui descendent en dessous de la ligne rouge sont des variations statistiquement significatives ($>5\sigma$).

**Ce que tu vas observer avec ce code :Sur le plot Science :**
- Les lignes pointillées te montrent la "profondeur" de chaque filtre sur cet objet précis.
- Sur le plot Diff : C'est là que c'est puissant. Si un point de la bande g (vert) descend plus bas que sa petite ligne pointillée verte, tu as une détection significative. S'il reste au-dessus, c'est juste du bruit de soustraction.
- Comparaison inter-bandes : Tu verras que les lignes $5\sigma$ ne sont pas au même niveau. La bande u aura probablement sa ligne "plus haute" (magnitude plus faible) que la bande r, car le ciel y est plus bruité ou le temps d'exposition différent.
- Un petit conseil visuel :Si les lignes pointillées surchargent trop le graphique, tu peux remplacer ax.axhline(...) par un petit segment coloré uniquement au début ou à la fin du graphique, ou simplement afficher la valeur moyenne de la limite dans la légende.

In [ ]:
assert False

### Pourquoi cette courbe est parfaite pour vos données DIA :
- Le passage par zéro : Contrairement à la courbe rouge (log), la courbe bleue traverse l'origine de manière fluide. Vous pouvez donc traiter un flux de $-2$ nJy et un flux de $+2$ nJy sans erreur mathématique.
- La symétrie : Si votre bruit de fond est gaussien (centré sur 0), $asinh$ préserve cette symétrie visuelle.
- La compression : Dès que $x$ devient grand (positif ou négatif), la pente s'adoucit. Cela "écrase" les échelles comme une magnitude le ferait, vous permettant de voir sur le même graphique des étoiles très brillantes et des fluctuations du fond de ciel.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Définition de la plage de données (incluant les valeurs négatives)
x = np.linspace(-10, 10, 500)
y = np.arcsinh(x)

# Comparaison avec log10 (uniquement pour x > 0)
x_log = np.linspace(0.1, 10, 250)
y_log = np.log10(x_log)

plt.figure(figsize=(10, 6))

# Tracé de asinh(x)
plt.plot(x, y, label=r"$y = arcsinh(x)$", color="blue", lw=2.5)

# Tracé de log10(x) pour comparaison
plt.plot(x_log, y_log, label=r"$y = log_{10}(x)$", color="red", linestyle="--", alpha=0.7)

# Mise en forme
plt.axhline(0, color="black", lw=1)
plt.axvline(0, color="black", lw=1)
plt.grid(True, alpha=0.3)

plt.title("Comportement de la fonction arcsinh(x)", fontsize=14, fontweight="bold")
plt.xlabel("Flux (x)", fontsize=12)
plt.ylabel("Valeur transformée (y)", fontsize=12)

# Annotations pour expliquer l'intérêt en astronomie
plt.annotate(
    "Linéaire à travers zéro",
    xy=(0, 0),
    xytext=(-8, 2),
    arrowprops=dict(facecolor="black", shrink=0.05, width=1, headwidth=5),
)
plt.annotate(
    "Comportement logarithmique",
    xy=(7, 2.6),
    xytext=(4, 4),
    arrowprops=dict(facecolor="black", shrink=0.05, width=1, headwidth=5),
)

plt.legend()
plt.show()

**Voulez-vous que j'ajuste ce tracé pour montrer exactement comment la constante $b$ (le paramètre d'adoucissement) modifie la largeur de la zone linéaire ?**

C'est une excellente idée. Visualiser l'impact de $b$ (le "softening parameter") est crucial, car c'est lui qui définit la frontière entre le régime où vos données se comportent comme un flux (linéaire) et celui où elles se comportent comme une magnitude (logarithmique).Voici un graphique comparatif pour illustrer ce basculement :

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Génération des données de flux (x)
x = np.linspace(-50, 50, 1000)

# On teste différentes valeurs de b (le niveau de bruit supposé)
b_values = [1, 5, 15]
labels = [
    "b = 1 : Zone linéaire étroite (Bruit faible)",
    "b = 5 : Zone linéaire intermédiaire",
    "b = 15 : Zone linéaire large (Bruit élevé)",
]
colors = ["#e63946", "#457b9d", "#1d3557"]

plt.figure(figsize=(12, 7))


for b, label, color in zip(b_values, labels, colors):
    # Formule simplifiée de la luptitude (sans le zero point pour la démo)
    # y = arcsinh(x / 2b)
    y = np.arcsinh(x / (2 * b))
    plt.plot(x, y, label=label, color=color, lw=2)

# Mise en forme
plt.axhline(0, color="black", lw=1, alpha=0.5)
plt.axvline(0, color="black", lw=1, alpha=0.5)
plt.grid(True, alpha=0.2, linestyle="--")

# Zones d'explication
plt.fill_between([-10, 10], -4, 4, color="gray", alpha=0.1, label="Zone de transition (Bruit)")

plt.title("Impact du paramètre 'b' sur la transformation asinh", fontsize=14, fontweight="bold")
plt.xlabel("Flux (nJy)", fontsize=12)
plt.ylabel("asinh(Flux / 2b)", fontsize=12)
plt.legend()

plt.annotate(
    "Régime Logarithmique (Magnitude)",
    xy=(40, 3.5),
    xytext=(25, 4.5),
    arrowprops=dict(arrowstyle="->", lw=1.5),
)
plt.annotate("Régime Linéaire (Flux)", xy=(2, 0.5), xytext=(-20, 2), arrowprops=dict(arrowstyle="->", lw=1.5))

plt.show()

**Ce qu'il faut retenir pour tes courbes de lumière :**

- `Si b est petit (courbe rouge)` : La fonction devient logarithmique très vite. C'est bien pour les sources très brillantes, mais le bruit près de zéro sera "étiré" violemment, ce qui peut rendre la courbe de lumière illisible si le fond de ciel est bruité.
  
- `Si b est grand (courbe foncée)` : La zone linéaire est beaucoup plus large. On voit mieux la distribution statistique du bruit (les points négatifs et positifs autour de zéro), mais on perd l'aspect "magnitude" pour les sources moyennement brillantes.

- `Le choix optimal` ($b \approx \sigma_{flux}$) : En prenant la médiane de l'erreur (FFERR_COL), on s'assure que la transition se fait exactement là où le signal commence à émerger du bruit. C'est le "sweet spot" mathématique.

- Petite astuce pour ton code Dual Plot :Dans la fonction plot_dia_object_dual_lup
titudes, j'ai calculé un $b$ différent pour le flux Science et le flux DIA.

- C'est correct car :Le flux Science inclut le bruit de l'image directe.

- Le flux DIA inclut le bruit de l'image directe ET du template.Le bruit DIA est donc naturellement plus élevé ($b_{diff} \approx \sqrt{2} \cdot b_{sci}$), et la Luptitude s'y adapte automatiquement.


Souhaites-tu que je t'aide à ajouter une ligne horizontale sur tes plots pour indiquer la magnitude limite à $5\sigma$ calculée à partir de ce paramètre $b$ ?

---
## 11. TBD Later : Coordinate Cross-Match with a Fink Alert

In [ ]:
FINK_RA = 53.125
FINK_DEC = -28.100
XMATCH_RADIUS_ARCSEC = 1.0

alert_sky = SkyCoord(ra=FINK_RA * u.deg, dec=FINK_DEC * u.deg)
sep_arcsec = alert_sky.separation(
    SkyCoord(ra=df_dia_obj_cone[RA_COL].values * u.deg, dec=df_dia_obj_cone[DEC_COL].values * u.deg)
).arcsec

df_xmatch = df_dia_obj_cone.copy()
df_xmatch["_sep_arcsec"] = sep_arcsec
df_xmatch_cut = (
    df_xmatch[df_xmatch["_sep_arcsec"] <= XMATCH_RADIUS_ARCSEC]
    .sort_values("_sep_arcsec")
    .reset_index(drop=True)
)

print(f'Candidates within {XMATCH_RADIUS_ARCSEC}": {len(df_xmatch_cut)}')
if len(df_xmatch_cut) > 0:
    show = [c for c in [OBJ_ID_COL, RA_COL, DEC_COL, "_sep_arcsec", NSRC_COL] if c in df_xmatch_cut.columns]
    display(df_xmatch_cut[show].head(10))

In [ ]:
if len(df_xmatch_cut) > 0 and len(df_forced_rich) > 0 and FOBJ_COL and FMJD_COL:
    best_id = df_xmatch_cut.iloc[0][OBJ_ID_COL]
    best_sep = df_xmatch_cut.iloc[0]["_sep_arcsec"]
    df_lc = df_forced_rich[df_forced_rich[FOBJ_COL] == best_id].sort_values(FMJD_COL)

    nrows_fig = 2 if (FDIFF_COL and FDIFF_COL in df_lc.columns) else 1
    fig, axes = plt.subplots(nrows_fig, 1, figsize=(16, 3 * nrows_fig), sharex=True)
    if nrows_fig == 1:
        axes = [axes]

    ax = axes[0]
    for band, grp in df_lc.groupby(FBAND_COL):
        yerr = grp[FFERR_COL].values if FFERR_COL else None
        ax.errorbar(
            grp[FMJD_COL],
            grp[FFLUX_COL],
            yerr=yerr,
            fmt="o",
            ms=4,
            alpha=0.85,
            capsize=2,
            color=BAND_COLORS.get(str(band), "grey"),
            label=str(band),
        )
    ax.axhline(0, color="grey", lw=0.8, ls="--")
    ax.set(
        ylabel="PSF flux (nJy)",
        title=f'Cross-match LC — {OBJ_ID_COL}={best_id}  sep={best_sep:.3f}"\n'
        f"Fink: RA={FINK_RA:.5f} deg  Dec={FINK_DEC:+.5f} deg",
    )
    ax.legend(title="Band", fontsize=9)
    ax.grid(True, alpha=0.3)

    if nrows_fig == 2:
        ax = axes[1]
        for band, grp in df_lc.groupby(FBAND_COL):
            yerr = grp[FDERR_COL].values if FDERR_COL else None
            ax.errorbar(
                grp[FMJD_COL],
                grp[FDIFF_COL],
                yerr=yerr,
                fmt="o",
                ms=4,
                alpha=0.85,
                capsize=2,
                color=BAND_COLORS.get(str(band), "grey"),
                label=str(band),
            )
        ax.axhline(0, color="grey", lw=0.8, ls="--")
        ax.set(ylabel="Diff PSF flux (nJy)")
        ax.legend(title="Band", fontsize=9)
        ax.grid(True, alpha=0.3)

    axes[-1].set_xlabel("MJD (TAI)", fontsize=11)

    # Add calendar date axis on top
    add_date_top_axis(axes[0], n_ticks=6)

    plt.tight_layout()
    plt.savefig(f"{DIR_FIGS}/XmatchLC_{best_id}_{SELECTED_DDF}.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("Cross-match LC skipped.")

---
## 12. Save Results

In [ ]:
for df, name in [
    (df_dia_obj_rich, f"DiaObjects_{SELECTED_DDF}_nmin{MIN_NSOURCES}.csv"),
    (df_dia_src_rich, f"DiaSources_{SELECTED_DDF}_nmin{MIN_NSOURCES}.csv"),
    (df_forced_rich, f"ForcedSrc_{SELECTED_DDF}_nmin{MIN_NSOURCES}.csv"),
]:
    if len(df) > 0:
        path = f"{DIR_DATA}/{name}"
        df.to_csv(path, index=False)
        print(f"Saved: {path}  ({len(df):,} rows)")

In [ ]:
# =========================================================================
# Helper: add a top x-axis showing calendar dates above the MJD axis
# =========================================================================
# The bottom x-axis of each light curve panel uses MJD.
# This helper adds a twin axis on top with evenly-spaced date labels
# (UTC) so you can immediately read off known observing dates.
#
# Usage (call AFTER plotting, BEFORE tight_layout / savefig):
#   add_date_top_axis(axes[0], n_ticks=6)
# =========================================================================

from astropy.time import Time as AstropyTime


def add_date_top_axis(ax, n_ticks=6, date_fmt="%Y-%m-%d"):
    """
    Add a secondary x-axis on top of `ax` showing calendar dates.

    The secondary axis shares the same MJD limits as the primary axis
    but labels ticks as UTC calendar dates (e.g. 2025-06-15).

    Parameters
    ----------
    ax : matplotlib Axes
        Primary axes whose bottom x-axis carries MJD values.
    n_ticks : int
        Number of evenly-spaced tick marks on the top axis.
    date_fmt : str
        strftime format for the date labels (default YYYY-MM-DD).

    Returns
    -------
    ax_top : matplotlib Axes
        The new twin axes placed on top.
    """
    mjd_min, mjd_max = ax.get_xlim()

    # Build evenly-spaced MJD tick positions spanning the current x range
    mjd_ticks = np.linspace(mjd_min, mjd_max, n_ticks)

    # Convert each MJD tick to a calendar date string via astropy
    date_labels = [AstropyTime(m, format="mjd", scale="utc").to_value("isot")[:10] for m in mjd_ticks]

    # Create a twin axis that shares the same x-scale
    ax_top = ax.twiny()
    ax_top.set_xlim(mjd_min, mjd_max)  # must match primary axis exactly
    ax_top.set_xticks(mjd_ticks)
    ax_top.set_xticklabels(date_labels, rotation=30, ha="left", fontsize=8)
    ax_top.set_xlabel("Date (UTC)", fontsize=9, labelpad=4)

    return ax_top


print("add_date_top_axis() helper defined.")